In [87]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules

# --- 1. CONFIGURAZIONE E CARICAMENTO DATI ---
print("--- 1. Caricamento e Ridenominazione ---")

# Carica i dataset
df_originale = pd.read_csv('./Dati_puliti/BankChurners_senza_predizione_NB.csv')
df_binned = pd.read_csv('continous_feature_binned.csv')

# Dizionario di mappatura per RINOMINARE le colonne originali in nomi brevi
column_rename_map = {
    # Categoriche Originali
    'Attrition_Flag': 'Stato_Cliente',
    'Gender': 'Genere',
    'Education_Level': 'Istruzione',
    'Marital_Status': 'Stato_Civile',
    'Income_Category': 'Reddito',
    'Card_Category': 'Tipo_Carta',
    
    # Numeriche Intere
    'Dependent_count': 'N_Persone_a_carico',
    'Total_Relationship_Count': 'N_Prodotti_finanziari',
    'Months_Inactive_12_mon': 'Mesi_Inattivi',
    'Contacts_Count_12_mon': 'Contatti_12M',
    
    # Float Binnate (rinominiamo le colonne binnate)
    'Customer_Age_bin': 'Età',
    'Months_on_book_bin': 'Anzianità_Clientela',
    'Credit_Limit_bin': 'Lim_Credito',
    'Total_Revolving_Bal_bin': 'Saldo_Revolving',
    'Avg_Open_To_Buy_bin': 'Disponibilità_residua', # Avg_Open_To_Buy
    'Total_Amt_Chng_Q4_Q1_bin': 'Var_Spesa_Q4Q1',
    'Total_Trans_Amt_bin': 'Importo_Transazioni',
    'Total_Trans_Ct_bin': 'N_Transazioni',
    'Total_Ct_Chng_Q4_Q1_bin': 'Var_N_Transazioni',
    'Avg_Utilization_Ratio_bin': 'Rapporto_Utilizzo'
}

# Applica la ridenominazione alle colonne
df_binned = df_binned.rename(columns={k: column_rename_map[k] for k in df_binned.columns})
df_fpgrowth_readable = df_originale.rename(columns=column_rename_map)

# Liste aggiornate dei nomi delle colonne
new_cat_cols = [v for k, v in column_rename_map.items() if k in df_originale.columns]
new_int_cols = [v for k, v in column_rename_map.items() if k in df_originale.columns and 'bin' not in k]
new_binned_cols = list(df_binned.columns)
all_new_cols = new_cat_cols + new_binned_cols


--- 1. Caricamento e Ridenominazione ---


In [88]:
# --- 2. ETICHETTATURA DEI VALORI (PER LA LEGGIBILITÀ) ---
print("--- 2. Etichettatura dei Valori ---")

# Dizionario di mappatura per sostituire gli intervalli (float binnati) con etichette chiare
# !!! VERIFICA e MODIFICA QUESTE ETICHETTE IN BASE AI TUOI BIN REALI !!!
bin_labels_mapping = {
    # 6 Bin: Età
    'Età': [
        'Giovane', 'Giovane-Medio', 'Media', 'Medio-Tarda', 'Tarda', 'Anziana'
    ],
    
    # 9 Bin: Anzianità_Clientela
    'Anzianità_Clientela': [
        'Nuovo_Cliente', 'Molto_Recente', 'Recente', 'Stabile_Bassa', 'Stabile_Media', 
        'Stabile_Alta', 'Fidelizzato_Basso', 'Fidelizzato_Medio', 'Fidelizzato_Alto'
    ],
    
    # 7 Bin: Lim_Credito
    'Lim_Credito': [
        'Standard_Basso', 'Standard_Medio', 'Standard_Alto', 'Premium_Basso', 
        'Premium_Medio', 'Premium_Alto', 'Elite'
    ],
    
    # 6 Bin: Saldo_Revolving
    'Saldo_Revolving': [
        'Molto_Basso', 'Basso', 'Medio', 'Medio-Alto', 'Alto', 'Massimo'
    ],
    
    # 6 Bin: AOTB (Avg Open To Buy - Credito Disponibile)
    'Disponibilità_residua': [
        'Minima', 'Bassa', 'Media', 'Medio-Alta', 'Alta', 'Molto_Alta'
    ],
    
    # 4 Bin: Var_Spesa_Q4Q1 (già corretta, ma la riincludo)
    'Var_Spesa_Q4Q1': [
        'Minima', 'Bassa', 'Media', 'Alta'
    ],
    
    # 8 Bin: Importo_Transazioni
    'Importo_Transazioni': [
        'Minime', 'Molto_Basse', 'Basse', 
        'Medie', 'Medio-Alte', 'Alte', 
        'Molto_Alte', 'VIP'
    ],
    
    # 8 Bin: N_Transazioni
    'N_Transazioni': [
        'Freq_Molto_Rara', 'Freq_Rara', 'Freq_Bassa', 'Freq_Medio-Bassa',
        'Freq_Media', 'Freq_Medio-Alta', 'Freq_Alta', 'Freq_Massima'
    ],
    
    # 4 Bin: Var_N_Transazioni (già corretta, ma la riincludo)
    'Var_N_Transazioni': [
        'Minima', 'Bassa', 'Media', 'Alta'
    ],
    
    # 4 Bin: Rapporto_Utilizzo (già corretta, ma la riincludo)
    'Rapporto_Utilizzo': [
        'Basso', 'Medio', 'Alto', 'Pieno'
    ]
}

# --- A. Gestione e Mappatura delle Variabili Float Binnate ---
for col in new_binned_cols:
    original_bins = sorted(df_binned[col].unique().astype(str).tolist())
    
    if col in bin_labels_mapping and len(original_bins) == len(bin_labels_mapping[col]):
        mapping = {original_bin: new_label for original_bin, new_label in zip(original_bins, bin_labels_mapping[col])}
    else:
        # Fallback se le etichette non corrispondono al numero di bin
        mapping = {original_bin: f"Bin_{i+1}" for i, original_bin in enumerate(original_bins)}
        
    df_fpgrowth_readable[col] = df_binned[col].astype(str).map(mapping).astype(str)

# --- B. Gestione delle Variabili Categoriche e Intere ---
for col in new_int_cols:
    # Per le colonne numeriche intere, usiamo il valore come etichetta
    # Aggiungiamo solo una descrizione chiara per 'Dipendenti'
    if col == 'Dipendenti':
        df_fpgrowth_readable[col] = df_fpgrowth_readable[col].astype(str).apply(lambda x: f"{x}_Dipendenti")
    else:
        df_fpgrowth_readable[col] = df_fpgrowth_readable[col].astype(str)
        
for col in new_cat_cols:
    # Le variabili categoriche sono già stringhe
    df_fpgrowth_readable[col] = df_fpgrowth_readable[col].astype(str)

# Combina tutte le colonne etichettate in un unico DataFrame finale
df_final = df_fpgrowth_readable[all_new_cols]


--- 2. Etichettatura dei Valori ---


In [89]:
df_final


,Stato_Cliente,Genere,Istruzione,Stato_Civile,Reddito,Tipo_Carta,N_Persone_a_carico,N_Prodotti_finanziari,Mesi_Inattivi,Contatti_12M,Età,Anzianità_Clientela,Lim_Credito,Saldo_Revolving,Disponibilità_residua,Var_Spesa_Q4Q1,Importo_Transazioni,N_Transazioni,Var_N_Transazioni,Rapporto_Utilizzo
0,Existing Customer,M,High School,Married,$60K - $80K,Blue,3,5,1,3,Media,Stabile_Alta,Elite,Alto,Alta,Alta,Molto_Alte,Freq_Molto_Rara,Bassa,Basso
1,Existing Customer,F,Graduate,Single,Less than $40K,Blue,5,6,1,2,Media,Fidelizzato_Basso,Premium_Alto,Massimo,Alta,Alta,Molto_Alte,Freq_Massima,Alta,Medio
2,Existing Customer,M,Graduate,Married,$80K - $120K,Blue,3,4,1,0,Medio-Tarda,Stabile_Media,Premium_Basso,Molto_Basso,Bassa,Alta,Minime,Freq_Massima,Media,Basso
3,Existing Customer,F,High School,Unknown,Less than $40K,Blue,4,3,4,1,Giovane-Medio,Stabile_Media,Premium_Basso,Medio-Alto,Media,Alta,Molto_Alte,Freq_Massima,Media,Pieno
4,Existing Customer,M,Uneducated,Married,$60K - $80K,Blue,3,5,1,0,Giovane-Medio,Molto_Recente,Premium_Medio,Molto_Basso,Medio-Alta,Alta,Molto_Alte,Freq_Massima,Media,Basso
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10122,Existing Customer,M,Graduate,Single,$40K - $60K,Blue,2,3,2,3,Medio-Tarda,Stabile_Alta,Premium_Medio,Medio,Bassa,Bassa,VIP,Freq_Alta,Minima,Alto
10123,Attrited Customer,M,Unknown,Divorced,$40K - $60K,Blue,2,4,2,3,Giovane-Medio,Recente,Premium_Medio,Medio-Alto,Bassa,Media,VIP,Freq_Medio-Bassa,Minima,Alto
10124,Attrited Customer,F,High School,Married,Less than $40K,Blue,1,5,3,4,Media,Stabile_Media,Premium_Medio,Molto_Basso,Medio-Alta,Media,VIP,Freq_Bassa,Minima,Basso
10125,Attrited Customer,M,Graduate,Unknown,$40K - $60K,Blue,2,4,3,3,Giovane,Stabile_Media,Premium_Medio,Molto_Basso,Medio-Alta,Minima,VIP,Freq_Bassa,Minima,Basso


In [90]:
# --- 3. CORREZIONE DEL PROBLEMA DEL DOPPIO PREFISSO E ONE-HOT ENCODING ---

# NON AGGIUNGIAMO IL PREFISSO QUI! Lasciamo che pd.get_dummies lo faccia una volta sola.
# df_final è pronto con i nomi delle colonne (es. 'Genere') e i valori (es. 'M').

# One-Hot Encoding: usa il nome della colonna come prefisso (es. 'Genere_')
# e il valore come suffisso (es. 'M'). Risultato: 'Genere_M'.
df_encoded_readable = pd.get_dummies(df_final, dtype=bool)

print("Etichette finali pronte. Esempio di itemsets (prime 5 colonne):", list(df_encoded_readable.columns[:5]))



Etichette finali pronte. Esempio di itemsets (prime 5 colonne): ['Stato_Cliente_Attrited Customer', 'Stato_Cliente_Existing Customer', 'Genere_F', 'Genere_M', 'Istruzione_College']


In [91]:
df_encoded_readable 

,Stato_Cliente_Attrited Customer,Stato_Cliente_Existing Customer,Genere_F,Genere_M,Istruzione_College,Istruzione_Doctorate,Istruzione_Graduate,Istruzione_High School,Istruzione_Post-Graduate,Istruzione_Uneducated,...,N_Transazioni_Freq_Molto_Rara,N_Transazioni_Freq_Rara,Var_N_Transazioni_Alta,Var_N_Transazioni_Bassa,Var_N_Transazioni_Media,Var_N_Transazioni_Minima,Rapporto_Utilizzo_Alto,Rapporto_Utilizzo_Basso,Rapporto_Utilizzo_Medio,Rapporto_Utilizzo_Pieno
0,False,True,False,True,False,False,False,True,False,False,...,True,False,False,True,False,False,False,True,False,False
1,False,True,True,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,True,False
2,False,True,False,True,False,False,True,False,False,False,...,False,False,False,False,True,False,False,True,False,False
3,False,True,True,False,False,False,False,True,False,False,...,False,False,False,False,True,False,False,False,False,True
4,False,True,False,True,False,False,False,False,False,True,...,False,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10122,False,True,False,True,False,False,True,False,False,False,...,False,False,False,False,False,True,True,False,False,False
10123,True,False,False,True,False,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,False
10124,True,False,True,False,False,False,False,True,False,False,...,False,False,False,False,False,True,False,True,False,False
10125,True,False,False,True,False,False,True,False,False,False,...,False,False,False,False,False,True,False,True,False,False


In [92]:
# Assumi che 'df_encoded_readable' sia il DataFrame generato dal passaggio 3 del codice completo

print("--- Elenco Completo degli Itemsets (Colonne) nel DataFrame Codificato ---")
print("-" * 60)

# Ottieni l'indice delle colonne come lista
itemsets_list = list(df_encoded_readable.columns)

# Itera e stampa ogni nome di colonna
for i, col_name in enumerate(itemsets_list):
    # Stampa l'indice (opzionale) e il nome della colonna (l'itemset)
    print(f"{i+1:3d}. {col_name}")

print("-" * 60)
print(f"Totale Colonne (Itemsets): {len(itemsets_list)}")

--- Elenco Completo degli Itemsets (Colonne) nel DataFrame Codificato ---
------------------------------------------------------------
  1. Stato_Cliente_Attrited Customer
  2. Stato_Cliente_Existing Customer
  3. Genere_F
  4. Genere_M
  5. Istruzione_College
  6. Istruzione_Doctorate
  7. Istruzione_Graduate
  8. Istruzione_High School
  9. Istruzione_Post-Graduate
 10. Istruzione_Uneducated
 11. Istruzione_Unknown
 12. Stato_Civile_Divorced
 13. Stato_Civile_Married
 14. Stato_Civile_Single
 15. Stato_Civile_Unknown
 16. Reddito_$120K +
 17. Reddito_$40K - $60K
 18. Reddito_$60K - $80K
 19. Reddito_$80K - $120K
 20. Reddito_Less than $40K
 21. Reddito_Unknown
 22. Tipo_Carta_Blue
 23. Tipo_Carta_Gold
 24. Tipo_Carta_Platinum
 25. Tipo_Carta_Silver
 26. N_Persone_a_carico_0
 27. N_Persone_a_carico_1
 28. N_Persone_a_carico_2
 29. N_Persone_a_carico_3
 30. N_Persone_a_carico_4
 31. N_Persone_a_carico_5
 32. N_Prodotti_finanziari_1
 33. N_Prodotti_finanziari_2
 34. N_Prodotti_finanziar

Per l'FP-growth, è necessario che ogni riga rappresenti una transazione e ogni colonna un item con un valore binario (presenza/assenza). Il formato di input standard è una matrice booleana.

Variabili Binnate e Categoriche Originali: Sono già stringhe (categoriche). Le codificheremo usando il One-Hot Encoding (o get_dummies di pandas) che è l'approccio più idoneo per l'FP-growth, poiché crea una colonna binaria per ogni valore unico all'interno di una colonna originale.

Variabili Numeriche Intere (int_cols): Anche queste le tratteremo come categoriche e applicheremo il One-Hot Encoding. Sebbene potremmo usare l'Ordinal Encoding, l'FP-growth non sfrutta l'ordine e il One-Hot Encoding ci permette di trattare ogni conteggio/numero intero come un item distinto, massimizzando il dettaglio dei pattern.

In [93]:
#!pip install mlxtend

In [94]:
# Scegli una soglia minima di supporto (min_support)
# min_support è la frequenza minima affinché un pattern sia considerato "frequente".
# La soglia dipenderà dalle dimensioni del dataset. Iniziamo con 1% (0.01)
min_support_threshold = 0.01

# Esegui l'algoritmo FP-growth
frequent_itemsets = fpgrowth(df_encoded_readable, min_support=min_support_threshold, use_colnames=True)

# Ordina i risultati per supporto decrescente
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)

print("\n--- Pattern Frequenti (min_support = {}) ---".format(min_support_threshold))
print(frequent_itemsets.head(50))


--- Pattern Frequenti (min_support = 0.01) ---
         support                                           itemsets  length
0       0.931767                                  (Tipo_Carta_Blue)       1
54      0.881011                         (Var_N_Transazioni_Minima)       1
1       0.839340                  (Stato_Cliente_Existing Customer)       1
116763  0.818604        (Var_N_Transazioni_Minima, Tipo_Carta_Blue)       2
107     0.781772  (Stato_Cliente_Existing Customer, Tipo_Carta_B...       2
108     0.728844  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       2
109     0.676015  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       3
20      0.529081                                         (Genere_F)       1
36753   0.503703                        (Genere_F, Tipo_Carta_Blue)       2
2       0.470919                                         (Genere_M)       1
36755   0.467562               (Genere_F, Var_N_Transazioni_Minima)       2
3       0.462822                        

In [95]:
from mlxtend.frequent_patterns import association_rules

# Estraiamo le regole, usando 'frequent_itemsets' come input
# Scegliamo una soglia minima di confidenza (min_threshold)
min_confidence_threshold = 0.7  # Ad esempio, almeno il 70% di probabilità

rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=min_confidence_threshold
)

# Ordina i risultati per Confidenza e Lift (che viene calcolato automaticamente)
rules = rules.sort_values(['confidence', 'lift'], ascending=[False, False])

#print("\n--- Regole di Associazione (Confidenza >= 0.7) ---")
#print(rules.head())

In [97]:
import pandas as pd
# Da eseguire solo se non l'hai già fatto:
# from mlxtend.frequent_patterns import association_rules

# Assumendo che 'rules' sia il DataFrame delle regole di associazione ordinato
# ... (codice per generare 'rules' omesso per brevità) ...

def print_formatted_rules(rules_df, num_rules_to_print=10):
    """
    Cicla sulle regole di associazione e stampa ciascuna in un formato dettagliato
    e leggibile, come richiesto.

    Args:
        rules_df (pd.DataFrame): Il DataFrame delle regole di associazione.
        num_rules_to_print (int): Il numero massimo di regole da stampare.
    """
    
    print("\n" + "="*80)
    print(f"STAMPA DETTAGLIATA DELLE PRIME {num_rules_to_print} REGOLE DI ASSOCIAZIONE")
    print("="*80 + "\n")

    # Contatore per il numero di regole stampate
    rule_counter = 0

    # Itera attraverso le righe del DataFrame delle regole
    for index, row in rules_df.head(num_rules_to_print).iterrows():
        if rule_counter >= num_rules_to_print:
            break

        rule_counter += 1

        print(f"Regola Numero {rule_counter}:")
        print("-" * 25)

        # --- ANTECEDENTE (IF) ---
        
        # Converte il frozenset in una lista di stringhe per la stampa
        antecedents_list = list(row['antecedents'])
        print("\nANTECEDENTE (IF):")
        
        # Intestazione per le variabili dell'antecedente
        print("  Variabili:")
        for item in antecedents_list:
            print(f"    - {item}")
        
        # --- CONSEQUENTE (THEN) ---

        # Converte il frozenset in una lista di stringhe per la stampa
        consequents_list = list(row['consequents'])
        print("\nCONSEGUENTE (THEN):")
        
        # Intestazione per le variabili del conseguente
        print("  Variabili:")
        for item in consequents_list:
            print(f"    - {item}")

        # --- INDICI (METRICHE) ---
        print("\nINDICI DI ASSOCIAZIONE:")
        
        # Stampa le metriche chiave in un formato ordinato
        # Si assumono le metriche standard generate da mlxtend
        
        print(f"  Supporto:   {row['support']:.4f}  (Frequenza congiunta del pattern)")
        print(f"  Confidenza: {row['confidence']:.4f}  (Probabilità che IF -> THEN)")
        print(f"  Lift:       {row['lift']:.4f}  (Quanto è migliore di una co-occorrenza casuale)")
        print(f"  Leverage:   {row['leverage']:.4f}  (Differenza tra frequenza congiunta e attesa)")
        print(f"  Conviction: {row['conviction']:.4f}  (Relazione con la Confidenza)")

        # Stampa la linea tratteggiata come separatore
        if rule_counter < num_rules_to_print:
            print("\n\n---\n\n")


# Esempio di utilizzo (stampa le prime 20 regole più forti)
# Assicurati di passare il tuo DataFrame 'rules'
print_formatted_rules(rules, num_rules_to_print=20)


STAMPA DETTAGLIATA DELLE PRIME 20 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Stato_Cliente_Existing Customer
    - Lim_Credito_Premium_Medio
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Rapporto_Utilizzo_Basso
    - Disponibilità_residua_Medio-Alta
    - Tipo_Carta_Blue

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0222  (Frequenza congiunta del pattern)
  Confidenza: 1.0000  (Probabilità che IF -> THEN)
  Lift:       18.1813  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0210  (Differenza tra frequenza congiunta e attesa)
  Conviction: inf  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Var_N_Transazioni_Minima
    - Stato_Cliente_Existing Customer
    - Lim_Credito_Premium_Medio
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Rapporto_Utilizzo_Basso
    - Disponibilità_residua_Medio-

In [98]:
# rules

In [55]:
# Esempio: Salvare il DataFrame delle Regole di Associazione (rules)
df_da_salvare = rules 

# Salva il DataFrame in un file CSV
df_da_salvare.to_csv(
    'regole_associazione_conf_more_70p_e_sup_min_1p.csv',  # Nome del file di output
    sep=';',                             # **Usa il punto e virgola come separatore**
    index=False,                         # Non includere l'indice di riga di Pandas
    encoding='utf-8'                     # Codifica standard per supportare caratteri speciali
)

In [2]:
import pandas as pd

# per ricaricaricare i frequent pattern al 1% di sup min e > 70 % confidenza
# Carica il file CSV nel DataFrame 'rules'
rules = pd.read_csv(
    'regole_associazione_conf_more_70p_e_sup_min_1p.csv',  # Nome del file
    sep=';',                                               # Usa il punto e virgola come separatore
    encoding='utf-8'                                       # Codifica per caratteri speciali
)

rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"frozenset({'Saldo_Revolving_Molto_Basso', 'Sta...","frozenset({'Tipo_Carta_Blue', 'Disponibilità_r...",0.022218,0.055001,0.022218,1.0,18.181329,1.0,0.020996,inf,0.966471,0.403950,1.000000,0.701975
1,"frozenset({'Lim_Credito_Premium_Medio', 'Saldo...","frozenset({'Tipo_Carta_Blue', 'Disponibilità_r...",0.018860,0.055001,0.018860,1.0,18.181329,1.0,0.017823,inf,0.963164,0.342908,1.000000,0.671454
2,"frozenset({'Saldo_Revolving_Molto_Basso', 'Lim...","frozenset({'Tipo_Carta_Blue', 'Disponibilità_r...",0.015108,0.055001,0.015108,1.0,18.181329,1.0,0.014277,inf,0.959495,0.274686,1.000000,0.637343
3,"frozenset({'Lim_Credito_Premium_Medio', 'Saldo...","frozenset({'Tipo_Carta_Blue', 'Disponibilità_r...",0.013923,0.055001,0.013923,1.0,18.181329,1.0,0.013157,inf,0.958342,0.253142,1.000000,0.626571
4,"frozenset({'Saldo_Revolving_Molto_Basso', 'Lim...","frozenset({'Tipo_Carta_Blue', 'Disponibilità_r...",0.013726,0.055001,0.013726,1.0,18.181329,1.0,0.012971,inf,0.958150,0.249551,1.000000,0.624776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657410,"frozenset({'Contatti_12M_1', 'Disponibilità_re...",frozenset({'Tipo_Carta_Blue'}),0.018762,0.931767,0.013133,0.7,0.751261,1.0,-0.004348,0.227445,-0.252295,0.014010,-3.396671,0.357047
657411,"frozenset({'Disponibilità_residua_Minima', 'Ge...",frozenset({'Tipo_Carta_Blue'}),0.015799,0.931767,0.011060,0.7,0.751261,1.0,-0.003662,0.227445,-0.251727,0.011809,-3.396671,0.355935
657412,"frozenset({'Disponibilità_residua_Minima', 'Ge...",frozenset({'Tipo_Carta_Blue'}),0.014812,0.931767,0.010368,0.7,0.751261,1.0,-0.003433,0.227445,-0.251538,0.011075,-3.396671,0.355564
657413,"frozenset({'Lim_Credito_Standard_Medio', 'Rapp...",frozenset({'Tipo_Carta_Blue'}),0.014812,0.931767,0.010368,0.7,0.751261,1.0,-0.003433,0.227445,-0.251538,0.011075,-3.396671,0.355564


Ottima osservazione! Avere oltre 65.000 regole di associazione è un risultato comune ma inutilizzabile. Hai assolutamente bisogno di strategie di filtraggio per trovare quelle più significative per il business.

La risposta alla tua domanda sul supporto minimo e le indicazioni per filtrare le regole sono collegate.

1. Strategie per Trovare Regole Indicative (Filtri)
Per focalizzarti sulle regole più utili, devi filtrare il tuo DataFrame rules usando combinazioni di metriche e di item specifici.

A. Filtro Basato sulle Metriche (Qualità)

Le due metriche più importanti per identificare regole "forti" e "interessanti" sono Confidenza e Lift.

Metrica	Significato	Filtro Consigliato	Focus Aziendale
Confidenza	Probabilità che il conseguente si verifichi dato l'antecedente ($P(B	A)$).	Alta (e.g., ≥0.7 o ≥0.8)
Lift	Quanto la co-occorrenza è migliore di quanto atteso per caso.	Alto (e.g., ≥1.5 o ≥2.0)	Regole Interessanti; indica una forte correlazione positiva non casuale.

In [108]:
# Filtra le regole per alta Confidenza E alto Lift
regole_forti = rules[
    (rules['confidence'] >= 0.7) & 
    (rules['lift'] >= 1.0)
]

print(f"Regole Forti (Conf. >= 0.7 e Lift >= 1.5): {len(regole_forti)}")

Regole Forti (Conf. >= 0.7 e Lift >= 1.5): 531370


In [109]:
# Filtra per regole che predicono l'abbandono
regole_churn = regole_forti[
    regole_forti['consequents'].apply(
        lambda x: 'Stato_Cliente_Attrited Customer' in str(x)
    )
]
regole_churn = regole_churn.sort_values(by='lift', ascending=False)
print(f"Regole Focus Churn (Consequente = Attrited Customer): {len(regole_churn)}")

Regole Focus Churn (Consequente = Attrited Customer): 3019


In [110]:
print_formatted_rules(regole_churn, num_rules_to_print=29)


STAMPA DETTAGLIATA DELLE PRIME 29 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Importo_Transazioni_Molto_Basse
    - Disponibilità_residua_Molto_Alta
    - Tipo_Carta_Blue
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Lim_Credito_Standard_Basso
    - Stato_Cliente_Attrited Customer

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0102  (Frequenza congiunta del pattern)
  Confidenza: 0.8655  (Probabilità che IF -> THEN)
  Lift:       22.8265  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0097  (Differenza tra frequenza congiunta e attesa)
  Conviction: 7.1555  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Importo_Transazioni_Molto_Basse
    - Disponibilità_residua_Molto_Alta
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Lim_Credito_Standard_Basso
    - Stato_Cliente_Attrited Customer
    -

# filtraggio delle regole per obbiettivi di business

In [111]:
# Filtra le regole che suggeriscono un upgrade a Silver o superiore (escludendo Blue)
regole_upgrade_carta = regole_forti[
    regole_forti['consequents'].apply(
        lambda x: any(item in str(x) for item in ['Tipo_Carta_Silver', 'Tipo_Carta_Gold', 'Tipo_Carta_Platinum'])
    )
].sort_values(by='lift', ascending=False)

regole_upgrade_carta  = regole_upgrade_carta .sort_values(by='lift', ascending=False)
print(f"Regole Focus Tipologia Carta (Consequente = Silver, Gold, Platinum): {len(regole_upgrade_carta)}")

Regole Focus Tipologia Carta (Consequente = Silver, Gold, Platinum): 6


In [103]:
print_formatted_rules(regole_upgrade_carta, num_rules_to_print=6)


STAMPA DETTAGLIATA DELLE PRIME 6 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Var_N_Transazioni_Minima
    - Lim_Credito_Elite
    - Reddito_Less than $40K

CONSEGUENTE (THEN):
  Variabili:
    - Tipo_Carta_Silver
    - Disponibilità_residua_Alta

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0117  (Frequenza congiunta del pattern)
  Confidenza: 0.7152  (Probabilità che IF -> THEN)
  Lift:       45.5493  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0114  (Differenza tra frequenza congiunta e attesa)
  Conviction: 3.4555  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Var_N_Transazioni_Minima
    - Lim_Credito_Elite
    - Reddito_Less than $40K
    - Disponibilità_residua_Alta

CONSEGUENTE (THEN):
  Variabili:
    - Tipo_Carta_Silver

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0117  (Frequenza congiunta del pattern)
  Confidenza: 0.7421  (Probab

In [112]:
# Filtra le regole dove l'antecedente o il conseguente contiene una forte inattività
regole_inattività = regole_forti[
    regole_forti.apply(
        lambda row: 'Mesi_Inattivi_5' in str(row['antecedents']) or 'Mesi_Inattivi_6' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_inattività  = regole_inattività.sort_values(by='lift', ascending=False)
print(f"Regole Focus Inattività della carta(Consequente = Mesi di inattività 5 o 6): {len(regole_inattività)}")

Regole Focus Inattività della carta(Consequente = Mesi di inattività 5 o 6): 10


In [113]:
print_formatted_rules(regole_inattività, num_rules_to_print=10)


STAMPA DETTAGLIATA DELLE PRIME 10 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Genere_F
    - Mesi_Inattivi_5

CONSEGUENTE (THEN):
  Variabili:
    - Tipo_Carta_Blue

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0104  (Frequenza congiunta del pattern)
  Confidenza: 0.9722  (Probabilità che IF -> THEN)
  Lift:       1.0434  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0004  (Differenza tra frequenza congiunta e attesa)
  Conviction: 2.4564  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Mesi_Inattivi_5

CONSEGUENTE (THEN):
  Variabili:
    - Var_N_Transazioni_Minima
    - Tipo_Carta_Blue

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0148  (Frequenza congiunta del pattern)
  Confidenza: 0.8427  (Probabilità che IF -> THEN)
  Lift:       1.0294  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0004  (Differenza tra frequenza congiun

In [115]:
# Filtra le regole dove il conseguente contiene 'Transazioni_Totali_Basso' o 'Transazioni_Totali_Medio'
regole_transazioni_basse_medie = regole_forti[
    regole_forti.apply(
        lambda row: 'Transazioni_Totali_Basso' in str(row['consequents']) or 'Transazioni_Totali_Medio' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_transazioni_basse_medie = regole_transazioni_basse_medie.sort_values(by='lift', ascending=False)
print(f"Regole Focus Transazioni Totali (Conseguente = Basso/Medio): {len(regole_transazioni_basse_medie)}")

Regole Focus Transazioni Totali (Conseguente = Basso/Medio): 0


In [116]:
# Filtra le regole dove il conseguente contiene una delle etichette del Saldo Revolving Totale
# Sostituisci 'Alto', 'Medio', 'Basso' con i nomi esatti delle tue colonne binarizzate
regole_revolving_bal = regole_forti[
    regole_forti.apply(
        lambda row: 'Tot_Revolving_Bal_Alto' in str(row['consequents']) or 'Tot_Revolving_Bal_Basso' in str(row['consequents']) or 'Tot_Revolving_Bal_Medio' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_revolving_bal = regole_revolving_bal.sort_values(by='lift', ascending=False)
print(f"Regole Focus Saldo Revolving (Consequente = Qualsiasi Fascia): {len(regole_revolving_bal)}")

Regole Focus Saldo Revolving (Consequente = Qualsiasi Fascia): 0


In [117]:
# Filtra le regole dove il conseguente è l'Ammontare Totale delle Transazioni Alto
regole_spesa_elevata = regole_forti[
    regole_forti.apply(
        lambda row: 'Ammontare_Transazioni_Totali_Alto' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_spesa_elevata = regole_spesa_elevata.sort_values(by='lift', ascending=False)
print(f"Regole Focus Valore Cliente (Consequente = Alto Ammontare Transazioni): {len(regole_spesa_elevata)}")

Regole Focus Valore Cliente (Consequente = Alto Ammontare Transazioni): 0


In [118]:
# Filtra le regole dove il conseguente contiene una delle etichette di variazione (change)
# Usiamo 'Change' come parola chiave, ipotizzando colonne come 'Trans_Amt_Change_Ultimi_12_Mesi_Basso' o 'Trans_Count_Change_Ultimi_12_Mesi_Alto'
regole_variazioni = regole_forti[
    regole_forti.apply(
        lambda row: any('Change' in item for item in row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_variazioni = regole_variazioni.sort_values(by='lift', ascending=False)
print(f"Regole Focus Variazioni (Consequente = Variazione Transazioni/Ammontare): {len(regole_variazioni)}")

Regole Focus Variazioni (Consequente = Variazione Transazioni/Ammontare): 0


In [119]:
# filtro generale da applicare alla fine 
# Aggiungi una colonna con la lunghezza combinata (antecedente + conseguente)
rules['length'] = rules['antecedents'].apply(lambda x: len(x)) + rules['consequents'].apply(lambda x: len(x))

# Filtra solo le regole di lunghezza media (es. da 2 a 5 item totali)
regole_filtrate = rules[
    (rules['length'] >= 2) & 
    (rules['length'] <= 5)
]

print(f"Regole filtrate per dimensione): {len(regole_filtrate)}")

Regole filtrate per dimensione): 327843


# Riproviamo il frequent pattern con FP grpwth aumentando il min_sup

In [84]:
# Scegli una soglia minima di supporto (min_support)
# min_support è la frequenza minima affinché un pattern sia considerato "frequente".
# La soglia dipenderà dalle dimensioni del dataset. Iniziamo con 1% (0.01)
min_support_threshold02 = 0.02

# Esegui l'algoritmo FP-growth
frequent_itemsets_minsup02 = fpgrowth(df_encoded_readable,
                                      min_support=min_support_threshold02, 
                                      use_colnames=True)

# Ordina i risultati per supporto decrescente
frequent_itemsets_minsup02['length'] = frequent_itemsets_minsup02['itemsets'].apply(lambda x: len(x))
frequent_itemsets_minsup02= frequent_itemsets_minsup05.sort_values(by='support', ascending=False)

print("\n--- Pattern Frequenti (min_support = {}) ---".format(min_support_threshold02))
print(frequent_itemsets_minsup02.head(50))


--- Pattern Frequenti (min_support = 0.02) ---
        support                                           itemsets  length
0      0.931767                                  (Tipo_Carta_Blue)       1
54     0.881011                         (Var_N_Transazioni_Minima)       1
1      0.839340                  (Stato_Cliente_Existing Customer)       1
12455  0.818604        (Var_N_Transazioni_Minima, Tipo_Carta_Blue)       2
102    0.781772  (Stato_Cliente_Existing Customer, Tipo_Carta_B...       2
103    0.728844  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       2
104    0.676015  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       3
20     0.529081                                         (Genere_F)       1
3952   0.503703                        (Genere_F, Tipo_Carta_Blue)       2
2      0.470919                                         (Genere_M)       1
3954   0.467562               (Genere_F, Var_N_Transazioni_Minima)       2
3      0.462822                             (Stato_C

In [85]:
# Estraiamo le regole, usando 'frequent_itemsets' come input
# Scegliamo una soglia minima di confidenza (min_threshold)
#min_confidence_threshold = 0.7  # Ad esempio, almeno il 70% di probabilità
min_confidence_threshold = 0.7

rules_02 = association_rules(
    frequent_itemsets_minsup02,
    metric="confidence",
    min_threshold=min_confidence_threshold
)

# Ordina i risultati per Confidenza e Lift (che viene calcolato automaticamente)
rules_02 = rules_02.sort_values(['confidence', 'lift'], ascending=[False, False])

In [34]:
# Scegli una soglia minima di supporto (min_support)
# min_support è la frequenza minima affinché un pattern sia considerato "frequente".
# La soglia dipenderà dalle dimensioni del dataset. Iniziamo con 1% (0.01)
min_support_threshold05 = 0.03

# Esegui l'algoritmo FP-growth
frequent_itemsets_minsup05 = fpgrowth(df_encoded_readable,
                                      min_support=min_support_threshold05, 
                                      use_colnames=True)

# Ordina i risultati per supporto decrescente
frequent_itemsets_minsup05['length'] = frequent_itemsets_minsup05['itemsets'].apply(lambda x: len(x))
frequent_itemsets_minsup05= frequent_itemsets_minsup05.sort_values(by='support', ascending=False)

print("\n--- Pattern Frequenti (min_support = {}) ---".format(min_support_threshold05))
print(frequent_itemsets_minsup05.head(50))


--- Pattern Frequenti (min_support = 0.03) ---
        support                                           itemsets  length
0      0.931767                                  (Tipo_Carta_Blue)       1
54     0.881011                         (Var_N_Transazioni_Minima)       1
1      0.839340                  (Stato_Cliente_Existing Customer)       1
12455  0.818604        (Var_N_Transazioni_Minima, Tipo_Carta_Blue)       2
102    0.781772  (Stato_Cliente_Existing Customer, Tipo_Carta_B...       2
103    0.728844  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       2
104    0.676015  (Var_N_Transazioni_Minima, Stato_Cliente_Exist...       3
20     0.529081                                         (Genere_F)       1
3952   0.503703                        (Genere_F, Tipo_Carta_Blue)       2
2      0.470919                                         (Genere_M)       1
3954   0.467562               (Genere_F, Var_N_Transazioni_Minima)       2
3      0.462822                             (Stato_C

In [35]:
from mlxtend.frequent_patterns import association_rules

# Estraiamo le regole, usando 'frequent_itemsets' come input
# Scegliamo una soglia minima di confidenza (min_threshold)
#min_confidence_threshold = 0.7  # Ad esempio, almeno il 70% di probabilità
min_confidence_threshold = 0.7

rules_05 = association_rules(
    frequent_itemsets_minsup05,
    metric="confidence",
    min_threshold=min_confidence_threshold
)

# Ordina i risultati per Confidenza e Lift (che viene calcolato automaticamente)
rules_05 = rules_05.sort_values(['confidence', 'lift'], ascending=[False, False])

In [36]:
print_formatted_rules(rules_05, num_rules_to_print=10)


STAMPA DETTAGLIATA DELLE PRIME 10 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Lim_Credito_Premium_Medio
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Rapporto_Utilizzo_Basso
    - Disponibilità_residua_Medio-Alta

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0349  (Frequenza congiunta del pattern)
  Confidenza: 1.0000  (Probabilità che IF -> THEN)
  Lift:       18.1163  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0329  (Differenza tra frequenza congiunta e attesa)
  Conviction: inf  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Lim_Credito_Premium_Medio
    - Tipo_Carta_Blue
    - Saldo_Revolving_Molto_Basso

CONSEGUENTE (THEN):
  Variabili:
    - Rapporto_Utilizzo_Basso
    - Disponibilità_residua_Medio-Alta

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0348  (Frequenza congiunta del pattern)
  Confidenza: 1.0000 

## Filtraggio delle regole associative 

In [73]:
# Filtra le regole per alta Confidenza E alto Lift
regole_forti_05 = rules_05[
    (rules_05['confidence'] >= 0.7) & 
    (rules_05['lift'] >= 1.0)
]

print(f"Regole Forti (Conf. >= 0.7 e Lift >= 1.5): {len(regole_forti_05)}")

Regole Forti (Conf. >= 0.7 e Lift >= 1.5): 39608


In [53]:
# Filtra per regole che predicono l'abbandono
regole_churn_05 = regole_forti_05[
    regole_forti_05['consequents'].apply(
        lambda x: 'Stato_Cliente_Attrited Customer' in str(x)
    )
]
regole_churn_05 = regole_churn_05.sort_values(by='lift', ascending=False)
print(f"Regole Focus Churn (Consequente = Attrited Customer): {len(regole_churn_05)}")

Regole Focus Churn (Consequente = Attrited Customer): 65


In [47]:
print_formatted_rules(regole_churn_05, num_rules_to_print=16)


STAMPA DETTAGLIATA DELLE PRIME 16 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Importo_Transazioni_Molto_Basse
    - N_Transazioni_Freq_Molto_Rara
    - Var_N_Transazioni_Minima

CONSEGUENTE (THEN):
  Variabili:
    - Stato_Cliente_Attrited Customer
    - Tipo_Carta_Blue

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0331  (Frequenza congiunta del pattern)
  Confidenza: 0.8272  (Probabilità che IF -> THEN)
  Lift:       5.5146  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0271  (Differenza tra frequenza congiunta e attesa)
  Conviction: 4.9179  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Importo_Transazioni_Molto_Basse
    - N_Transazioni_Freq_Molto_Rara

CONSEGUENTE (THEN):
  Variabili:
    - Var_N_Transazioni_Minima
    - Stato_Cliente_Attrited Customer

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0340  (Frequenza congiunta del pattern)
  C

In [54]:
# Filtra le regole che suggeriscono un upgrade a Silver o superiore (escludendo Blue)
regole_upgrade_carta_05 = regole_forti_05[
    regole_forti_05['consequents'].apply(
        lambda x: any(item in str(x) for item in ['Tipo_Carta_Silver', 'Tipo_Carta_Gold', 'Tipo_Carta_Platinum'])
    )
].sort_values(by='lift', ascending=False)

regole_upgrade_carta_05  = regole_upgrade_carta_05.sort_values(by='lift', ascending=False)
print(f"Regole Focus Tipologia Carta (Consequente = Silver, Gold, Platinum): {len(regole_upgrade_carta_05)}")

Regole Focus Tipologia Carta (Consequente = Silver, Gold, Platinum): 0


In [55]:
# Filtra le regole dove l'antecedente o il conseguente contiene una forte inattività
regole_inattività_05 = regole_forti_05[
    regole_forti_05.apply(
        lambda row: 'Mesi_Inattivi_5' in str(row['antecedents']) or 'Mesi_Inattivi_6' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_inattività_05  = regole_inattività_05.sort_values(by='lift', ascending=False)
print(f"Regole Focus Inattività della carta(Consequente = Mesi di inattività 5 o 6): {len(regole_inattività_05)}")

Regole Focus Inattività della carta(Consequente = Mesi di inattività 5 o 6): 0


In [66]:
# Filtra le regole che iniziano con un reddito molto alto per vedere cosa fanno (consequente)
regole_alto_reddito_05 = regole_forti_05[
    regole_forti_05['antecedents'].apply(
        lambda x: 'Reddito_$120K +' in str(x)
    )
].sort_values(by='lift', ascending=False)

regole_alto_reddito_05  = regole_alto_reddito_05.sort_values(by='lift', ascending=False)
print(f"Regole Focus Reddito(Consequente = Alto): {len(regole_alto_reddito_05)}")

Regole Focus Reddito(Consequente = Alto): 8


In [67]:
print_formatted_rules(regole_alto_reddito_05, num_rules_to_print=8)


STAMPA DETTAGLIATA DELLE PRIME 8 REGOLE DI ASSOCIAZIONE

Regola Numero 1:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Disponibilità_residua_Minima
    - Stato_Cliente_Existing Customer
    - Reddito_$120K +

CONSEGUENTE (THEN):
  Variabili:
    - Lim_Credito_Standard_Medio
    - Genere_M

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0306  (Frequenza congiunta del pattern)
  Confidenza: 0.9254  (Probabilità che IF -> THEN)
  Lift:       7.3848  (Quanto è migliore di una co-occorrenza casuale)
  Leverage:   0.0265  (Differenza tra frequenza congiunta e attesa)
  Conviction: 11.7209  (Relazione con la Confidenza)


---


Regola Numero 2:
-------------------------

ANTECEDENTE (IF):
  Variabili:
    - Rapporto_Utilizzo_Basso
    - Disponibilità_residua_Minima
    - Reddito_$120K +

CONSEGUENTE (THEN):
  Variabili:
    - Lim_Credito_Standard_Medio
    - Genere_M

INDICI DI ASSOCIAZIONE:
  Supporto:   0.0354  (Frequenza congiunta del pattern)
  Confidenza: 0.9156  (Probabilit

In [70]:
# filtro generale da applicare alla fine 
# Aggiungi una colonna con la lunghezza combinata (antecedente + conseguente)
rules_05['length'] = rules_05['antecedents'].apply(lambda x: len(x)) + rules_05['consequents'].apply(lambda x: len(x))

# Filtra solo le regole di lunghezza media (es. da 2 a 5 item totali)
regole_filtrate_05 = rules_05[
    (rules_05['length'] >= 2) & 
    (rules_05['length'] <= 5)
]

regole_filtrate_05 = regole_filtrate_05.sort_values(by='lift', ascending=False)
print(f"Regole filtrate per numero items): {len(regole_filtrate_05)}")

Regole filtrate per numero items): 44686


In [75]:
# Filtra le regole dove il conseguente contiene 'Transazioni_Totali_Basso' o 'Transazioni_Totali_Medio'
regole_transazioni_basse_medie = regole_filtrate_05[
    regole_filtrate_05.apply(
        lambda row: 'Transazioni_Totali_Basso' in str(row['consequents']) or 'Transazioni_Totali_Medio' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_transazioni_basse_medie = regole_transazioni_basse_medie.sort_values(by='lift', ascending=False)
print(f"Regole Focus Transazioni Totali (Conseguente = Basso/Medio): {len(regole_transazioni_basse_medie)}")

Regole Focus Transazioni Totali (Conseguente = Basso/Medio): 0


In [76]:
# Filtra le regole dove il conseguente contiene una delle etichette del Saldo Revolving Totale
# Sostituisci 'Alto', 'Medio', 'Basso' con i nomi esatti delle tue colonne binarizzate
regole_revolving_bal = regole_filtrate_05[
    regole_filtrate_05.apply(
        lambda row: 'Tot_Revolving_Bal_Alto' in str(row['consequents']) or 'Tot_Revolving_Bal_Basso' in str(row['consequents']) or 'Tot_Revolving_Bal_Medio' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_revolving_bal = regole_revolving_bal.sort_values(by='lift', ascending=False)
print(f"Regole Focus Saldo Revolving (Consequente = Qualsiasi Fascia): {len(regole_revolving_bal)}")

Regole Focus Saldo Revolving (Consequente = Qualsiasi Fascia): 0


In [77]:
# Filtra le regole dove il conseguente contiene una delle etichette di variazione (change)
# Usiamo 'Change' come parola chiave, ipotizzando colonne come 'Trans_Amt_Change_Ultimi_12_Mesi_Basso' o 'Trans_Count_Change_Ultimi_12_Mesi_Alto'
regole_variazioni = regole_filtrate_05[
    regole_filtrate_05.apply(
        lambda row: any('Change' in item for item in row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_variazioni = regole_variazioni.sort_values(by='lift', ascending=False)
print(f"Regole Focus Variazioni (Consequente = Variazione Transazioni/Ammontare): {len(regole_variazioni)}")

Regole Focus Variazioni (Consequente = Variazione Transazioni/Ammontare): 0


In [78]:
# Filtra le regole dove l'antecedente contiene simultaneamente Basso Utilizzo Linea di Credito E Alto Numero di Contatti
regole_basso_uso_alto_contatto = regole_filtrate_05[
    regole_filtrate_05.apply(
        lambda row: 'Utilizzo_Linea_Credito_Basso' in str(row['antecedents']) and 
                   'Contatti_Ultimi_12_Mesi_Alto' in str(row['antecedents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_basso_uso_alto_contatto = regole_basso_uso_alto_contatto.sort_values(by='lift', ascending=False)
print(f"Regole Focus Contatto/Utilizzo (Antecedente = Basso Utilizzo + Alto Contatto): {len(regole_basso_uso_alto_contatto)}")

Regole Focus Contatto/Utilizzo (Antecedente = Basso Utilizzo + Alto Contatto): 0


In [79]:
# Filtra le regole dove il conseguente è l'Ammontare Totale delle Transazioni Alto
regole_spesa_elevata = regole_filtrate_05[
    regole_filtrate_05.apply(
        lambda row: 'Ammontare_Transazioni_Totali_Alto' in str(row['consequents']), axis=1
    )
].sort_values(by='confidence', ascending=False)

regole_spesa_elevata = regole_spesa_elevata.sort_values(by='lift', ascending=False)
print(f"Regole Focus Valore Cliente (Consequente = Alto Ammontare Transazioni): {len(regole_spesa_elevata)}")

Regole Focus Valore Cliente (Consequente = Alto Ammontare Transazioni): 0


## Filtraggio regole con minsup 2%


In [86]:
# Filtra le regole per alta Confidenza E alto Lift
regole_forti_02 = rules_02[
    (rules_02['confidence'] >= 0.7) & 
    (rules_02['lift'] >= 1.0)
]

print(f"Regole Forti (Conf. >= 0.7 e Lift >= 1.5): {len(regole_forti_02)}")

Regole Forti (Conf. >= 0.7 e Lift >= 1.5): 39608


Analisi risultati uguale a 3% di min sup inutile continuare